# Finding historical DIA light curves

In this notebook we will crossmatch alerts with the Data Preview 2 (DP2) DIA object data and visualize their forced source light curves.

1. Load the Rubin alert archive catalog and the DP2 DIA object collection.
2. Crossmatch the alerts against the DIA objects. Both catalogs carry forced photometry timeseries.
3. Combine `prvDiaForcedSources` (from each alert) with the matching `diaObjectForcedSource`.
4. Visualize resulting light curves with [`lsdb_rubin.plot_light_curve`](https://lsdb-rubin.readthedocs.io/en/latest/reference/plot_light_curve.html).

In [ ]:
import lsdb
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lsdb import ConeSearch

lsdb.show_versions()

We need `lsdb-rubin`, a package that contains a suite of utilities for interacting with Rubin LSST data within LSDB. Install it with:

In [ ]:
%pip install -q lsdb-rubin

Restart the kernel to apply the changes before proceeding.

## Load the catalogs

Let's define the paths to the data:

In [ ]:
ALERTS_PATH = "https://data.lsdb.io/hats/rubin_alert_archive"
DIA_OBJECT_PATH = "/rubin/lsdb_data/dp2/dia_object_collection"

For demonstration purposes, we will select a small region in the sky defined by the following cone:

In [ ]:
cone = ConeSearch(ra=52.838, dec=-28.279, radius_arcsec=3600)

Let's load both catalogs. It's important to specify the search region and the columns we need. This will keep the workflow light, especially in terms of memory usage.

In [ ]:
dia_object = lsdb.open_catalog(
    DIA_OBJECT_PATH, search_filter=cone, columns=["diaObjectId", "diaObjectForcedSource"]
)

In [ ]:
alerts = lsdb.open_catalog(ALERTS_PATH, search_filter=cone, columns=["diaSourceId", "prvDiaForcedSources"])
# Remove the alerts with no forced sources
alerts = alerts.query("prvDiaForcedSources.len() > 0")

**Note:** This version of the alert archive has data until June 29th and remains frozen for the time being. If you're interested in a specific alert ID, add the following`filters=[("diaSourceId", "=", <SOURCE_ID>)]` to the alerts `open_catalog` call.

## Crossmatch alerts to DIA objects

Next, we will match each alert to its nearest Rubin DP2 DIA object, within 1 arcsecond (the default). The columns we previously selected from both catalogs are carried into the result. `ra` and `dec` are suffixed with `_alert` and `_dia` since they overlap.

In [ ]:
xmatched = alerts.crossmatch(dia_object, suffix_method="overlapping_columns", suffixes=("_alert", "_dia"))
xmatched

We can preview a few matched rows with `Catalog.head`, which this triggers a small amount of work.

In [ ]:
preview = xmatched.head(4)
preview

## Plot the light curves

Let's visualize the matched light curves of our `preview` with [`lsdb_rubin.plot_light_curve`](https://lsdb-rubin.readthedocs.io/en/latest/reference/plot_light_curve.html). This utility method expects a flat table with `band`, `midpointMjdTai`, and a brightness field with a matching `<field>Err` column.

We are interested to plot `psfMag` over time but the alerts don't ship `prvDiaForcedSources` with magnitudes. Let's convert the fluxes to magnitudes and append them as new subcolumns of `prvDiaForcedSources`.

In [ ]:
def add_prv_mag(flux, flux_err):
    """Convert flux (nJy) to AB magnitude (asymmetric error from flux ± fluxErr)"""
    flux = np.asarray(flux, dtype="float64")
    flux_err = np.asarray(flux_err, dtype="float64")
    mag = u.nJy.to(u.ABmag, flux)
    upper = u.nJy.to(u.ABmag, flux + flux_err)
    lower = u.nJy.to(u.ABmag, flux - flux_err)
    mag_err = -(upper - lower) / 2
    return {"prvDiaForcedSources.psfMag": mag, "prvDiaForcedSources.psfMagErr": mag_err}


def append_prv_mag(nf):
    """Append psfMag/psfMagErr onto the nested prvDiaForcedSources column"""
    return nf.map_rows(
        add_prv_mag,
        columns=["prvDiaForcedSources.psfFlux", "prvDiaForcedSources.psfFluxErr"],
        append_columns=True,
        row_container="args",
    )


preview = append_prv_mag(preview)

Now both nested light curves expose `band`, `midpointMjdTai`, `psfMag`, and `psfMagErr`. Let's combine them and plot them.

In [ ]:
def combined_light_curve(match, i):
    """Concatenate both nested light curves, keeping band, time, and magnitude."""
    prv_dia_forced_source = match["prvDiaForcedSources"].iloc[i]
    forced_source = match["diaObjectForcedSource"].iloc[i]
    return pd.concat([prv_dia_forced_source, forced_source], ignore_index=True)

In [ ]:
from lsdb_rubin.plot_light_curve import plot_light_curve

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150)
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(preview):
        ax.set_visible(False)
        continue
    plt.sca(ax)
    plot_light_curve(
        combined_light_curve(preview, i),
        mag_field="psfMag",
        title=f"diaSourceId {preview["diaSourceId"].iloc[i]}",
    )

fig.tight_layout()
plt.show()

## About

**Authors:** Sandro Campos, Konstantin Malanchev

**Last updated/verified on:** Jul 21, 2026

If you use `lsdb` for published research, please cite following [instructions](https://docs.lsdb.io/en/stable/citation.html).